# Qwen3.5-0.8B | Colab A100 VLM Benchmark

This uploadable notebook runs **seven method arms** (Base, H2O+Base, Declarative Attention+Base, ECRD+Base, native MTP, H2O+MTP, DA+MTP). Base/MTP have native vLLM runners. The five external-code arms are **not yet ported or GPU-qualified** and will fail closed, not silently become Base of Qwen/Qwen3.5-0.8B, with MemLens and MMLongBench-Doc. It clones the public colab-vlm-bench repository automatically. It **does not** need a notebook connector or a separately uploaded Python helper.

**Run status:** All GPU switches default to False to prevent unintended A100 charges. A finished preflight/normalization is not a successful benchmark. To produce real serving receipts, first opt in to bounded GPU token annotation (Step 8), verify readiness (Step 9), then opt in to Base (Step 10). Do not enable unported methods.\n\n**Workflow:** configure Drive paths → A100 preflight → install vLLM → stage checkpoint on local SSD → prepare native VLM datasets → derive input-length serving samples → Base-server token annotation → Base then MTP benchmark → optional native-context quality → runbook export.

**Scientific split:** shortened derived prompts measure serving only; native full-context questions are used for quality. Official benchmark accuracy requires publishers' scorers. The notebook has not been GPU-executed by the author; unsupported MTP/context/VRAM cells must remain recorded as such.

**Before execution:** Colab Runtime → Change runtime type → A100 GPU. Inspect the README in the repository for required dataset layouts.

## 1. Edit Drive paths and switches

Modify paths below to locations you have actually uploaded. The model directory must contain checkpoint files (`.safetensors` or `.bin`), tokenizer and processor configs, not just a symbolic link. Start with all benchmark switches off; turn on annotation, Base, MTP and quality one at a time after the GPU checks succeed.

In [ ]:
from pathlib import Path
import os
import sys

MODEL_ID = "qwen35_08b"
HF_MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_DRIVE_DIR = "/content/drive/MyDrive/LLM_Weights/models/Qwen/Qwen3.5-0.8B"  # EDIT
MEMLENS_ROOT = "/content/drive/MyDrive/Research/Topic/Speculative Decoding/STARLIGHT/memlens"  # EDIT
MEMLENS_RAW_JSON = str(Path(MEMLENS_ROOT) / "dataset_32k.json")
MEMLENS_IMAGES_ROOT = MEMLENS_ROOT  # contains release_images/
MMLONG_ROOT = "/content/drive/MyDrive/Research/Topic/Speculative Decoding/STARLIGHT/mmlongbench_doc"  # EDIT
MMLONG_RAW_QA_JSON = str(Path(MMLONG_ROOT) / "data" / "train-00000-of-00001.parquet")  # EDIT: actual publisher QA JSON
MMLONG_PDF_ROOT = str(Path(MMLONG_ROOT) / "documents")  # EDIT
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/Research/Topic/Speculative Decoding/Stage 0 - Observation/results"  # EDIT if desired

# Pilot first; full = seven lengths x six CCUs x three repetitions for this model.
PHASE = "pilot"  # "pilot" or "full"
PREPARE_NATIVE = True
PREPARE_DERIVED = True
ACTIVE_DATASETS = ["memlens", "mmlongbench_doc"]  # can restrict to ["memlens"] until PDFs ready
RUN_GPU_ANNOTATION = False  # set True after Base server smoke; costs GPU time
RUN_BASE = False            # set True to execute serving cells for Base
RUN_MTP = False             # set True only after MTP compatibility is confirmed
RUN_NATIVE_QUALITY = False # native full-context predictions; may OOM on A100
RUN_PROXY_METRICS = False  # auxiliary token EM/F1, not official

# Seven-arm experiment: five external-method variants remain BLOCKED
# until actual Qwen3.5+A100 engine ports pass correctness tests.
RUN_METHOD_MATRIX = False
METHOD_SELECTION = ["base", "h2o_base", "da_base", "ecrd_base",
                    "mtp", "h2o_mtp", "da_mtp"]
RUN_HALLUCINATION_SCORING = False  # requires independent judge JSONL
DOWNLOAD_SUMMARY = False

ANNOTATION_CALL_BUDGET = 8  # maximum NEW one-token GPU calls per dataset per invocation
MAX_CELLS_PER_JOB = 6       # increase only after pilot timing/capacity checks
MAX_ROWS_PER_BIN = 192      # >=160 for offered CCU32 requests+warmups
MMLONG_NORMALIZE_LIMIT = 200  # pilot: cap QA/PDF rendering; full=0 for all 1,091 questions
MAX_IMAGES_PER_PROMPT = 16 # frozen per campaign; A100 may reject heavy native docs
VLLM_VERSION = "0.29.0"     # verify wheel/driver compatibility in this Colab session
INSTALL_DEPENDENCIES = True
CPU_PREFLIGHT_ONLY = False  # True on a free CPU Colab runtime; stop after Step 3b
assert MODEL_ID in ("qwen35_08b", "qwen35_4b")
assert PHASE in ("pilot", "full")
print("Selected:", MODEL_ID, "phase:", PHASE)


## 2. Mount Google Drive and verify the accelerator

If the runtime is not an A100 **40GB**, stop here. These experiments must not silently switch to T4/L4/H100/A100-80GB.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import subprocess, json
if CPU_PREFLIGHT_ONLY:
    print("CPU preflight only: A100 check deferred. Run through Step 3b and STOP.")
else:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader,nounits"], capture_output=True, text=True, check=True)
    print(gpu.stdout)
    parts = [x.strip() for x in gpu.stdout.strip().splitlines()[0].split(",")]
    assert len(gpu.stdout.strip().splitlines()) == 1 and "A100" in parts[0], "A100 GPU is required"
    assert 39000 <= float(parts[1]) <= 45000, "This campaign requires A100 40GB (MiB inventory)"
    print("A100 40GB preflight PASS. GPU allocation and runtime lifecycle remain under your control.")


## 3. Clone benchmark code and install dependencies

A Colab runtime may need a restart after installing vLLM, especially if it changes Torch/CUDA wheels. Never proceed after an unresolved import error. The repository revision is printed and stored in results.

In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = Path("/content/colab-vlm-bench")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/anatwork14/colab-vlm-bench.git", str(REPO)], check=True)
else:
    print("Refreshing existing checkout to main:", REPO)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)], check=True)
if INSTALL_DEPENDENCIES:
    pkgs = ["ijson>=3.3,<4", "pyarrow>=15,<25", "psutil>=5.9"]
    if not CPU_PREFLIGHT_ONLY:
        pkgs += ["vllm==" + VLLM_VERSION, "transformers>=4.57",
                 "PyMuPDF>=1.24", "pandas", "tqdm", "nbformat"]
    subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], check=True)
source_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                                      text=True).strip()
sys.path.insert(0, str(REPO))
print("Repository source SHA:", source_sha)
try:
    import importlib.metadata
    if not CPU_PREFLIGHT_ONLY:
        print("Installed vLLM:", importlib.metadata.version("vllm"))
        print("Installed transformers:", importlib.metadata.version("transformers"))
    else:
        print("CPU_ONLY: skipped heavy vLLM/CUDA installs")
except Exception as exc:
    raise RuntimeError("Dependency installation incomplete; resolve/restart runtime") from exc
if not CPU_PREFLIGHT_ONLY:
    assert importlib.metadata.version("vllm") == VLLM_VERSION


## 3b. CPU-only preflight: system RAM, 200GB SSD, source datasets and model

**Run this before staging weights, normalizing any PDF, or launching vLLM.** Use the *actual available* RAM/SSD rather than nominal 80GB RAM / 200GB disk. The gate reserves at least 10GiB free RAM and enough local SSD for the target checkpoint plus a safety margin (minimum 18GiB). It also reads only the first bytes of MemLens JSON and a small batch of MMLongBench QA Parquet, confirms sampled PDFs exist, and checks checkpoint layout **without loading model tensors**.

If one dataset is not yet available, restrict `ACTIVE_DATASETS` in cell 1 rather than silently skipping it. Results for excluded datasets must remain NOT_RUN. This is a CPU resource check, not a guarantee against model KV/OOM at CCU32.

In [ ]:
import sys
from pathlib import Path

# Ensure the cloned repository's source directories are in Python's search path
for path in ["/content/colab-vlm-bench", "/content/colab-vlm-bench/src"]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vlmbench.preflight import (
    host_resources, checkpoint_inventory, dataset_probe_memlens,
    dataset_probe_mmlong, assess_resources,
)
assert set(ACTIVE_DATASETS).issubset({"memlens", "mmlongbench_doc"})
assert ACTIVE_DATASETS, "Select at least one dataset"
assert PHASE in ("pilot", "full")
if PHASE == "full":
    MMLONG_NORMALIZE_LIMIT = 0  # full native quality cannot be clipped
checkpoint_info = checkpoint_inventory(Path(MODEL_DRIVE_DIR))
host_info = host_resources(Path("/content/vlmbench"))
resource_report = assess_resources(host_info, checkpoint_info["total_bytes"])
print("ACTUAL system RAM available (GiB):", host_info["ram_available_gib"])
print("ACTUAL local SSD free (GiB):", host_info["disk_free_gib"])
print("Checkpoint bytes:", checkpoint_info["total_bytes"])
print("RAM/SSD readiness:", resource_report["status"])
if resource_report["status"] != "CPU_PREFLIGHT_PASS":
    raise RuntimeError("Not enough RAM or SSD before GPU work: " + str(resource_report["issues"]))
if "memlens" in ACTIVE_DATASETS:
    print("MemLens source:", dataset_probe_memlens(
        Path(MEMLENS_RAW_JSON), Path(MEMLENS_IMAGES_ROOT)))
if "mmlongbench_doc" in ACTIVE_DATASETS:
    print("MMLong source:", dataset_probe_mmlong(
        Path(MMLONG_RAW_QA_JSON), Path(MMLONG_PDF_ROOT)))
print("CPU_PREFLIGHT_PASS: dataset layout checked; GPU inference still NOT_QUALIFIED")

## 4. Stage model weights from Drive onto local SSD

This step **does not download from Hugging Face**; it copies the checkpoint already on Drive, verifies SHA-256 of all staged files, and leaves the Drive originals untouched. vLLM will load the model from local SSD.

In [ ]:
if CPU_PREFLIGHT_ONLY:
    raise RuntimeError(
        "CPU preflight complete. STOP here, select A100 40GB, "
        "set CPU_PREFLIGHT_ONLY=False and rerun setup before staging weights."
    )
import sys
from pathlib import Path

# Ensure the source directory containing 'vlmbench' is appended to sys.path
REPO_SRC_DIR = "/content/colab-vlm-bench/src"
if REPO_SRC_DIR not in sys.path:
    sys.path.insert(0, REPO_SRC_DIR)
if "/content/colab-vlm-bench" not in sys.path:
    sys.path.insert(0, "/content/colab-vlm-bench")

import hashlib, json
from vlmbench.artifacts import staged_copy
from vlmbench.engine import a100_inventory

assert Path(MODEL_DRIVE_DIR).is_dir(), (
    "Edit MODEL_DRIVE_DIR to the existing Drive checkpoint directory: " + MODEL_DRIVE_DIR)
inventory = a100_inventory()
STAGED_MODEL = Path("/content/vlmbench/model_stage") / MODEL_ID / "target"
staged = staged_copy(MODEL_DRIVE_DIR, STAGED_MODEL)
weight_manifest_digest = hashlib.sha256(
    json.dumps(staged["files"], sort_keys=True).encode("utf8")).hexdigest()
MODEL_SOURCE_REVISION = "local-checkpoint-sha256-" + weight_manifest_digest[:20]
print("Staged model:", STAGED_MODEL)
print("Checkpoint files:", len(staged["files"]))
print("Pinned local source identity:", MODEL_SOURCE_REVISION)
print("GPU:", inventory)

## 5. Normalize real datasets (native quality track) — bounded RAM

MemLens 32K has 789 long-context questions. The normalizer streams one question at a time using ijson and writes JSONL incrementally; do **not** use json.load() / recursive role patching. The code accepts case-insensitive roles including "AI Assistant". Previously interrupted runs may leave a .pending file, but only completed native JSONL is treated as ready.

MMLongBench-Doc accepts a QA .json **or .parquet** file; pages are rendered incrementally (this can use considerable disk). A mismatch between QA columns and the pinned publisher schema fails explicitly, rather than inventing data. Already completed normalized outputs are reused.

In [ ]:
from pathlib import Path
# Discard any stale in-process monkey-patch from a previous Colab execution.
import importlib
import vlmbench.datasets as dataset_module
importlib.reload(dataset_module)
from vlmbench.datasets import normalize_memlens, normalize_mmlong
if "ijson" not in normalize_memlens.__code__.co_consts.__repr__():
    raise RuntimeError("Stale dataset normalizer loaded; restart Colab and re-run setup")

# RAM-safe: process ONE MemLens question at a time using ijson.
# Role mapping (user / AI Assistant) is handled in normalize_memlens itself.
# Never json.load() the whole 104MB source, recursively copy it, or write
# an intermediate patched_dataset_32k.json file.
DRIVE_INPUT_DIR = Path(DRIVE_RESULTS_ROOT) / "prepared" / MODEL_ID
DRIVE_INPUT_DIR.mkdir(parents=True, exist_ok=True)
NATIVE = {
    "memlens": DRIVE_INPUT_DIR / "memlens_native.jsonl",
    "mmlongbench_doc": DRIVE_INPUT_DIR / "mmlong_native.jsonl",
}
if PREPARE_NATIVE:
    if "memlens" in ACTIVE_DATASETS and not NATIVE["memlens"].is_file():
        assert Path(MEMLENS_RAW_JSON).is_file(), "MemLens dataset_32k.json missing on Drive"
        assert (Path(MEMLENS_IMAGES_ROOT) / "release_images").is_dir(), "MemLens images missing"
        print("MemLens: streaming normalization; only one question in RAM")
        print(normalize_memlens(
            Path(MEMLENS_RAW_JSON), Path(MEMLENS_IMAGES_ROOT), NATIVE["memlens"]))
    if "mmlongbench_doc" in ACTIVE_DATASETS and not NATIVE["mmlongbench_doc"].is_file():
        assert Path(MMLONG_RAW_QA_JSON).is_file(), (
            "MMLongBench QA file missing (.json or .parquet); edit the path")
        assert Path(MMLONG_PDF_ROOT).is_dir(), "MMLongBench PDF directory missing"
        print("MMLongBench: bounded PDF-page rendering with lazy QA reading")
        print(normalize_mmlong(
            Path(MMLONG_RAW_QA_JSON), Path(MMLONG_PDF_ROOT),
            DRIVE_INPUT_DIR / "mmlong_rendered_pages",
            NATIVE["mmlongbench_doc"],
            limit=(MMLONG_NORMALIZE_LIMIT or None), dpi=96))
for dataset, native_file in NATIVE.items():
    print(dataset, "READY" if native_file.is_file() else "MISSING", native_file)


## 6. Prepare source-derived real VLM serving prompts

This is a **different evaluation track** from native accuracy. It creates candidate text bins while retaining one real image from each source; it never copies gold answers into prompts. Pilot targets 2,048 and 16,384 model-tokenized text tokens; the full campaign includes all seven target bins. A candidate is not eligible for measured benchmarking until its multimodal prompt-token count is verified by the running Base vLLM endpoint.

In [ ]:
from vlmbench.notebook_bridge import build_serving_candidates
from vlmbench.config import LENGTHS
TEXT_BINS = [2048, 16384] if PHASE == "pilot" else list(LENGTHS)
CANDIDATES = {ds: DRIVE_INPUT_DIR / (ds + "_" + PHASE + "_candidates.jsonl")
              for ds in NATIVE}
ANNOTATED = {ds: DRIVE_INPUT_DIR / (ds + "_" + PHASE + "_annotated.jsonl")
             for ds in NATIVE}
if PREPARE_DERIVED:
    for dataset in ACTIVE_DATASETS:
        if not NATIVE[dataset].is_file():
            print(dataset, "SKIPPED: missing native JSONL")
            continue
        if CANDIDATES[dataset].exists():
            print(dataset, "using existing candidates", CANDIDATES[dataset])
            continue
        result = build_serving_candidates(
            NATIVE[dataset], CANDIDATES[dataset], dataset=dataset,
            model_directory=STAGED_MODEL, input_lengths=TEXT_BINS,
            max_per_bin=MAX_ROWS_PER_BIN,
            pdf_root=MMLONG_PDF_ROOT if dataset == "mmlongbench_doc" else None)
        print(json.dumps(result, indent=2))
for ds in NATIVE:
    print(ds, "candidates:", CANDIDATES[ds].exists(),
          "annotated:", ANNOTATED[ds].exists())


## 7. Freeze local campaign and backend parameters

Each notebook uses a separate immutable campaign ID so the two checkpoint sources cannot collide. Base and MTP within the same model use exactly the same config, dataset IDs, seeds and serving conditions. The repo's other models stay deferred.

In [ ]:
from vlmbench.notebook_bridge import write_local_campaign
from vlmbench.config import load_config
from vlmbench.engine import server_argv
from pathlib import Path

template_name = ("a100_qwen_priority_pilot.json" if PHASE == "pilot"
                 else "a100_qwen_priority_full.json")
TEMPLATE = REPO / "configs" / template_name
LOCAL_CONFIG = Path("/content/vlmbench") / ("campaign_" + MODEL_ID + "_" + PHASE + ".json")
DATA_PATHS = {
    ds: {"native": str(NATIVE[ds]), "derived": str(ANNOTATED[ds])}
    for ds in ("memlens", "mmlongbench_doc")
}
summary = write_local_campaign(
    TEMPLATE, LOCAL_CONFIG, model_id=MODEL_ID,
    drive_model=MODEL_DRIVE_DIR,
    drive_results=str(Path(DRIVE_RESULTS_ROOT) / "measurements"),
    data_paths=DATA_PATHS, revision=MODEL_SOURCE_REVISION)
cfg = load_config(LOCAL_CONFIG, execution=True)
cfg["serving"]["max_images_per_prompt"] = MAX_IMAGES_PER_PROMPT
cfg["serving"]["backend_version"] = VLLM_VERSION
# Frozen notebook-specific adjustments must be made BEFORE any GPU benchmark receipts.
LOCAL_CONFIG.write_text(json.dumps(cfg, indent=2) + "\n", encoding="utf8")
cfg = load_config(LOCAL_CONFIG, execution=True)
CAMPAIGN_ID = cfg["campaign_id"]
model_spec = next(x for x in cfg["models"] if x["id"] == MODEL_ID)
print("Campaign:", CAMPAIGN_ID, "planned cells:", summary["cell_count"])
print("Base launch:", server_argv(cfg, model_spec, "base", str(STAGED_MODEL)))
print("MTP launch:", server_argv(cfg, model_spec, "mtp", str(STAGED_MODEL)))
print("Per-job maximum:", MAX_CELLS_PER_JOB,
      "Note: only real GPU receipts count as benchmark results")


## Seven-arm source-pinned matrix and backend qualification

This cell freezes all seven methods, references the real H₂O/Declarative Attention/ECRD repositories, and checks **temperature=0, top_p=1.0**. The method planner does not allocate GPU. A stock vLLM Base server is **not H₂O/DA/ECRD**: those five arms require real Qwen3.5+A100 backend/kernel integration with trace evidence.

For two VLMs, pilot is **168 planned serving cells**, full **3,528**; this notebook handles only the selected checkpoint, so its share is 84 pilot / 1,764 full.

In [ ]:
from vlmbench.methods import (
    METHODS, UPSTREAM, plan as seven_arm_plan,
    manifest_from_path, native_mode_or_block, blocked_method_receipt,
)
method_file = REPO / "configs" / (
    "a100_qwen_7methods_pilot.json" if PHASE == "pilot"
    else "a100_qwen_7methods_full.json"
)
seven_manifest = manifest_from_path(method_file)
seven_summary = seven_arm_plan(seven_manifest)
assert seven_manifest["temperature"] == 0 and seven_manifest["top_p"] == 1.0
assert cfg["serving"]["temperature"] == 0 and cfg["serving"].get("top_p", 1.0) == 1.0
assert list(METHODS) == METHOD_SELECTION
print("Methods:", ", ".join(METHODS))
print("This model planned cells:", seven_summary["total_serving_cells"] // 2)
for entry in seven_summary["methods"]:
    print(entry["id"], entry["implementation_status"])
print("Pinned external sources:", {k: v["commit"] for k, v in UPSTREAM.items()})


## 8. Start Base server and annotate multimodal effective prompt tokens

Set `RUN_GPU_ANNOTATION=True` in cell 1, then execute this cell. It starts a real Base VLM server and tests an image request, annotates the derived prompts using one-token inference, and **stops the server** when done. This step is necessary before the fixed-length CCU benchmark. If annotation fails, do not mark unannotated bins as benchmark PASS.

In [ ]:
from vlmbench.engine import start_server, wait_ready, stop_server
from vlmbench.derivation import annotate_with_backend
from vlmbench.workloads import load_jsonl, resolve_messages
import httpx

if RUN_GPU_ANNOTATION:
    assert ANNOTATION_CALL_BUDGET > 0, "Start with bounded GPU calls"
    assert any(CANDIDATES[ds].is_file() for ds in ACTIVE_DATASETS), "No source-derived candidates"
    log = Path("/content/vlmbench/logs") / (MODEL_ID + "_base_annotation.log")
    argv = server_argv(cfg, model_spec, "base", str(STAGED_MODEL))
    proc = start_server(argv, log)
    try:
        wait_ready(proc, cfg["serving"]["port"], timeout_s=600, expected_model=MODEL_ID)
        print("Base VLM ready; performing one real image-request smoke test")
        first_ds = next(ds for ds in ACTIVE_DATASETS
                        if CANDIDATES[ds].is_file())
        first_row = load_jsonl(CANDIDATES[first_ds],
                              dataset=first_ds, track="derived_serving")[0]
        with httpx.Client(timeout=300) as client:
            payload = {
                "model": MODEL_ID, "messages": resolve_messages(first_row["messages"]),
                "max_tokens": 16, "temperature": 0, "top_p": 1.0, "stream": False,
                "chat_template_kwargs": {"enable_thinking": False},
            }
            response = client.post("http://127.0.0.1:8000/v1/chat/completions", json=payload)
            response.raise_for_status()
            obj = response.json()
            print("Image smoke PASS; server usage:", obj.get("usage", {}))
        for ds in ACTIVE_DATASETS:
            if not CANDIDATES[ds].is_file():
                print("SKIP annotation", ds, ": candidate JSONL missing")
                continue
            print("Annotating", ds, "with capped new requests:", ANNOTATION_CALL_BUDGET,
                  "(resumes prior verified tokens if annotated output exists)")
            print(annotate_with_backend(
                CANDIDATES[ds], ANNOTATED[ds], ds, MODEL_ID,
                str(STAGED_MODEL), "http://127.0.0.1:8000", limit=ANNOTATION_CALL_BUDGET))
    finally:
        stop_server(proc)
        print("Annotation Base server stopped; log:", log)
else:
    print("RUN_GPU_ANNOTATION=False. No paid annotation launched.")


## 9. Check dataset readiness and expected serving sample capacity

At CCU=32, each cell needs **160 unique usable prompts** (128 measured + 32 warmups). A real source may not have enough 16K text examples or may exceed 32K total after adding vision tokens. The benchmark must record `INSUFFICIENT_WORKLOAD` or capacity failures instead of silently duplicating prompts.

In [ ]:
from vlmbench.config import planned_cells
from vlmbench.preflight import serving_readiness
SERVING_READINESS = {}
for ds in ACTIVE_DATASETS:
    selected_cells = [x for x in planned_cells(cfg) if
                      x["model"] == MODEL_ID and x["dataset"] == ds
                      and x["mode"] == "base"]
    readiness = serving_readiness(
        ANNOTATED[ds], dataset=ds, model_id=MODEL_ID, cells=selected_cells,
        tolerance=cfg["performance"]["max_bin_relative_error"],
        max_model_len=cfg["serving"]["max_model_len"])
    SERVING_READINESS[ds] = readiness
    print(ds, readiness["status"], "eligible per text bin:",
          readiness.get("eligible_by_text_bin", {}))
    print("Ready:", len(readiness.get("ready_cells", [])),
          "Blocked:", readiness.get("blocked_cells", "?"))
print("ONLY enable RUN_BASE / RUN_MTP for datasets with READY_CELLS_PRESENT")

**Why the default notebook does not run:** `RUN_GPU_ANNOTATION`, `RUN_BASE`, and `RUN_MTP` are intentionally `False`. Change a switch in Step 1 **and re-execute Step 1** before running its GPU cell. Step 8 makes at most `ANNOTATION_CALL_BUDGET` new requests *per active dataset* per execution (default 8), then resumes the verified JSONL on later executions. A partial annotation is expected to leave Step 9 blocked; do not start paid serving until Step 9 reports `READY_CELLS_PRESENT` for that dataset. The count requirements depend on CCU and input bin; some bins may lack enough distinct source questions even after all annotations succeed. Keep those cells `INSUFFICIENT_WORKLOAD`/blocked rather than synthesizing samples. Never confuse the seven-arm plan with seven executable methods: external H₂O/DA/ECRD ports are not qualified.


## 10. Execute BASE serving benchmark

To run, set `RUN_BASE=True` in cell 1 and execute this cell. Starts/stops a fresh vLLM server using BF16 on A100, records TTFT/TPOT/E2E/output tok/s, GPU memory and per-cell failures, and synchronizes receipts to Google Drive. Resume is supported. **Output must actually contain 256 generated tokens to count as PASS.**

In [ ]:
from vlmbench.cli import main as vlmbench_cli
if RUN_BASE:
    for ds in ACTIVE_DATASETS:
        if not ANNOTATED[ds].is_file():
            print("SKIP Base:", ds, "requires annotated source-derived VLM prompts")
            continue
        print("RUN BASE", MODEL_ID, ds)
        vlmbench_cli(["run-job", "--config", str(LOCAL_CONFIG),
                      "--model", MODEL_ID, "--mode", "base", "--dataset", ds,
                      "--max-cells", str(MAX_CELLS_PER_JOB)])
else:
    print("RUN_BASE=False. No benchmark GPU job launched.")


## 11. Execute MTP SPECULATIVE DECODING serving benchmark

To run, set `RUN_MTP=True` in cell 1 and execute this cell. The current pinned backend uses `method=mtp` for Qwen3.5; the official Qwen model card examples show the older `qwen3_next_mtp` alias. This notebook **never falls back automatically** to Base or a different speculative method. Inspect the MTP server log/acceptance instrumentation before interpreting any speedup.

In [ ]:
if RUN_MTP:
    for ds in ACTIVE_DATASETS:
        if not ANNOTATED[ds].is_file():
            print("SKIP MTP:", ds, "requires annotated source-derived VLM prompts")
            continue
        print("RUN MTP", MODEL_ID, ds, "with", model_spec["mtp_tokens"], "speculative tokens")
        vlmbench_cli(["run-job", "--config", str(LOCAL_CONFIG),
                      "--model", MODEL_ID, "--mode", "mtp", "--dataset", ds,
                      "--max-cells", str(MAX_CELLS_PER_JOB)])
    print("Check the server logs in Drive for actual MTP activation before paper claims.")
else:
    print("RUN_MTP=False. No speculative GPU job launched.")


## Seven-method dispatch (fail closed for unported methods)

Set `RUN_METHOD_MATRIX=True` to walk the selected methods in order. It executes only genuine native vLLM Base/MTP arms via the existing benchmark runner. Other arms record `BACKEND_UNQUALIFIED` with source provenance and **no GPU metrics**. A research agent must implement and qualify each backend before changing this policy.

**Avoid duplicate Base/MTP runs:** use either the dedicated Base/MTP cells above, or the matrix dispatcher below; don't enable both routes for the same campaign.

In [ ]:
from vlmbench.artifacts import atomic_json
if RUN_METHOD_MATRIX:
    for method_name in METHOD_SELECTION:
        for ds in ACTIVE_DATASETS:
            if method_name in ("base", "mtp"):
                if RUN_BASE or RUN_MTP:
                    print("Skip duplicate native run", method_name,
                          "(RUN_BASE or RUN_MTP already selected)")
                    continue
                if not ANNOTATED[ds].is_file():
                    print("Missing annotated workload", method_name, ds)
                    continue
                print("EXECUTING NATIVE METHOD:", method_name, MODEL_ID, ds)
                vlmbench_cli([
                    "run-job", "--config", str(LOCAL_CONFIG),
                    "--model", MODEL_ID, "--mode",
                    native_mode_or_block(method_name), "--dataset", ds,
                    "--max-cells", str(MAX_CELLS_PER_JOB),
                ])
            else:
                blocked = blocked_method_receipt(method_name, MODEL_ID, ds)
                status_path = DRIVE_CAMPAIGN if "DRIVE_CAMPAIGN" in globals() else (
                    Path(cfg["storage"]["drive_results_dir"]) / CAMPAIGN_ID)
                status_path = status_path / "method_qualifications" / (
                    ds + "__" + MODEL_ID + "__" + method_name + ".json")
                atomic_json(status_path, blocked)
                print("BLOCKED (not a GPU benchmark):", method_name, ds,
                      "receipt:", status_path)
else:
    print("RUN_METHOD_MATRIX=False; external arms require independently qualified engines.")


## 12. Optional native full-context VLM quality

Set `RUN_NATIVE_QUALITY=True` only after serving qualification. This uses **full MemLens 32K history and full MMLongBench PDF-page image sequences**, at CCU=1 with a 65,536-token maximum model context (if supported). It can be infeasible on A100 40GB; do not truncate and claim official accuracy. Pilot processes a deterministic subset (up to 48 per dataset).

In [ ]:
if RUN_NATIVE_QUALITY:
    for mode in ("base", "mtp"):
        for ds in ACTIVE_DATASETS:
            if not NATIVE[ds].is_file():
                print("SKIP native quality", ds, "missing full-context input")
                continue
            print("NATIVE QUALITY", MODEL_ID, mode, ds)
            vlmbench_cli(["run-job", "--config", str(LOCAL_CONFIG),
                          "--model", MODEL_ID, "--mode", mode,
                          "--dataset", ds, "--quality"])
else:
    print("RUN_NATIVE_QUALITY=False (no full-context native predictions).")


## Grounded hallucination metrics — independent judge required

Official native accuracy/F1 remain publisher-specific. **Hallucination is not identical to an incorrect answer.** A separate independent judge must inspect the generated full-context answer and the corresponding evidence sessions/PDF pages, extract atomic factual claims, and assign SUPPORTED / UNSUPPORTED / CONTRADICTED / UNVERIFIABLE with evidence IDs. MemLens refusal and MMLongBench unanswerable items support false assertion and abstention metrics. See `docs/HALLUCINATION_EVALUATION.md` for the frozen rubric and `configs/hallucination_annotation.example.jsonl` for required schema.

When no judged label file exists, the metric is **NOT_MEASURED**, not zero. Turn on `RUN_HALLUCINATION_SCORING` only after annotation files have been independently produced.

In [ ]:
from vlmbench.hallucination import read_annotations, score_evidence_annotations
from vlmbench.quality import deterministic_subset
from vlmbench.artifacts import atomic_json
JUDGE_DIR = Path(DRIVE_RESULTS_ROOT) / "judged_annotations" / CAMPAIGN_ID
if RUN_HALLUCINATION_SCORING:
    for ds in ("memlens", "mmlongbench_doc"):
        if not NATIVE[ds].is_file():
            print("Native dataset missing:", ds)
            continue
        native_rows = load_jsonl(NATIVE[ds], dataset=ds, track="native_quality")
        native_rows = deterministic_subset(
            native_rows, int(cfg["quality"].get("max_samples", 0)),
            int(cfg["quality"].get("sample_seed", 42)))
        expected = [str(row["id"]) for row in native_rows]
        for arm in METHOD_SELECTION:
            file = JUDGE_DIR / (ds + "__" + MODEL_ID + "__" + arm + ".jsonl")
            if not file.is_file():
                print("NOT_MEASURED hallucination:", ds, arm)
                continue
            report = score_evidence_annotations(
                read_annotations(file), expected_ids=expected, require_complete=True)
            out = Path(cfg["storage"]["drive_results_dir"]) / CAMPAIGN_ID
            out = out / "quality" / (ds + "__" + MODEL_ID + "__" + arm + "_evidence.json")
            atomic_json(out, report)
            print("INDEPENDENT-JUDGE HALLUCINATION", ds, arm,
                  "answer_rate", report["answer_hallucination_rate"],
                  "unsupported_claim_rate", report["unsupported_claim_rate"])
else:
    print("RUN_HALLUCINATION_SCORING=False; hallucination NOT_MEASURED until judged.")


## 13. Read results, paired comparisons and optional proxy EM/F1

Measured serving results contain only observed requests. Proxy EM/token-F1 can help debugging the native prediction files but are **not** official MemLens judge accuracy or MMLongBench official F1.

In [ ]:
from vlmbench.report import write_report
from vlmbench.quality import score_proxy
import pandas as pd, json
ROOT = Path(cfg["storage"]["local_scratch_dir"]) / CAMPAIGN_ID
DRIVE_CAMPAIGN = Path(cfg["storage"]["drive_results_dir"]) / CAMPAIGN_ID
result = write_report(ROOT, cfg)
print("Serving cell statuses:", result["status_counts"])
print("Paired Base/MTP comparisons:", len(result.get("paired_comparisons", [])))
if result.get("paired_comparisons"):
    display(pd.DataFrame(result["paired_comparisons"])[
        ["dataset", "model", "input_length", "concurrency",
         "paired_repetitions", "throughput_speedup_median"]].head(30))
if RUN_PROXY_METRICS:
    from vlmbench.workloads import load_jsonl
    from vlmbench.artifacts import atomic_json
    for ds in NATIVE:
        if not NATIVE[ds].is_file():
            continue
        native = load_jsonl(NATIVE[ds], dataset=ds, track="native_quality")
        for mode in ("base", "mtp"):
            path = ROOT / "quality" / (ds + "__" + MODEL_ID + "__" + mode + ".jsonl")
            if not path.is_file():
                continue
            preds = [json.loads(x) for x in path.read_text(encoding="utf8").splitlines() if x]
            scores = score_proxy(preds, native)
            atomic_json(ROOT / "quality" / (ds + "__" + MODEL_ID + "__" + mode + "_proxy.json"), scores)
            print("AUXILIARY ONLY:", ds, mode, scores.get("token_f1"), scores.get("exact_match"))
from vlmbench.artifacts import sync_files
sync_files(ROOT, DRIVE_CAMPAIGN)
print("Runbook and raw receipts on Drive:", DRIVE_CAMPAIGN)


## 14. Export a small runbook bundle

All raw receipts/logs remain on Drive. Set `DOWNLOAD_SUMMARY=True` to download a small ZIP containing the observed summary and runbook. Do not upload credentials, checkpoint weights or private raw documents to GitHub.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files
bundle = Path("/content") / (CAMPAIGN_ID + "_runbook.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for name in ("RUNBOOK.md", "summary.json", "environment.json", "run_status.json"):
        item = ROOT / name
        if item.is_file():
            zipf.write(item, arcname=name)
print("Exported:", bundle)
print("Drive raw evidence:", DRIVE_CAMPAIGN)
if DOWNLOAD_SUMMARY:
    files.download(str(bundle))


## 15. Next step / audit

Repeat each serving run cell until the pilot's 24 cells for this model have a recorded status. Do not replace OOM/unsupported/missing workload with zero latency. Only compare Base and MTP after their model, dataset, exact sample-ID hashes, seeds and output-token policy match. For official accuracy: run upstream MemLens LLM-as-judge and MMLongBench-Doc scorer separately, record evaluator version and report the native sample denominator. For full study, switch PHASE to full and prepare all seven bins with a **new campaign ID**.